# User Review Scraper — Revolut, Klarna, Wise (last 12 months)

Scrapes **20,000 public user reviews** for **Revolut**, **Klarna** and **Wise** from both the
**Google Play Store** and the **Apple App Store**, filtered to the **past 12 months**.

**Libraries used**
- [`google-play-scraper`](https://pypi.org/project/google-play-scraper/) for Android reviews
- [`app-store-scraper`](https://pypi.org/project/app-store-scraper/) for iOS reviews

> **App Store note.** PyPI's `app-store-scraper` 0.3.5 calls Apple's internal `amp-api.apps.apple.com`,
> which now requires a signed JWT and returns `401 Unauthorized`. Cell 3 defines `RSSAppStore`, a thin
> subclass that reads the same data from Apple's public **RSS customer-reviews feed**
> (`itunes.apple.com/rss/customerreviews/...`). That feed is capped at **10 pages × 50 = 500 reviews per
> app**, so the App Store sources are limited to 500/app and **Google Play carries the rest of the 20,000**.
>
> **Country code note.** Apple's storefront code for the UK is **`gb`**, not `uk` — using `uk` makes the
> RSS endpoint return `HTTP 400` (a previous run silently collected 0 App Store reviews because of this).

## Environment

Run with the `fintech-app-reviews` kernel (project venv):

```bash
python -m venv .venv
.venv/bin/pip install google-play-scraper app-store-scraper pandas ipykernel
.venv/bin/python -m ipykernel install --user --name fintech-app-reviews
```


In [2]:
from datetime import datetime, timedelta
import json, os, time, re, random

import pandas as pd
from google_play_scraper import reviews as gp_reviews, Sort as GPSort
from app_store_scraper import AppStore

## 1. Configuration

`TARGET_TOTAL` is the number of reviews to collect overall. Allocation is **dynamic**:
- App Store is capped at `APP_STORE_CAP_PER_APP = 500` per app (RSS feed hard limit).
- Google Play receives the remainder, split evenly across the 3 apps, so the totals still sum to ~20,000.

Set `SMOKE_TEST = True` for a fast end-to-end check (~10 per source) before the full run.


In [3]:
TARGET_TOTAL = 20_000
MONTHS       = 12          # look-back window in months
SMOKE_TEST   = False       # set False for the full 20k run
COUNTRY      = "gb"        # Apple storefront code (UK = "gb", NOT "uk")
LANG         = "en"
DATA_DIR     = "data"
RAW_DIR      = os.path.join(DATA_DIR, "raw")
SLEEP_GPLAY  = 0.5         # seconds between Google Play pages
SLEEP_RSS    = 0.5         # seconds between App Store RSS pages
APP_STORE_CAP_PER_APP = 500   # RSS feed max (10 pages x 50)

os.makedirs(RAW_DIR, exist_ok=True)

CUTOFF = datetime.now() - timedelta(days=int(365 * MONTHS / 12))
print("Reviews older than", CUTOFF.date(), "are excluded")

APPS = {
    "Revolut": {"gplay": "com.revolut.revolut",    "ios_id": 932493382, "ios_name": "revolut"},
    "Klarna":  {"gplay": "com.myklarnamobile",     "ios_id": 1115120118, "ios_name": "klarna"},
    "Wise":    {"gplay": "com.transferwise.android", "ios_id": 612261027, "ios_name": "wise"},
}

SOURCES = [("google_play", app) for app in APPS] + [("app_store", app) for app in APPS]

# Dynamic allocation: App Store capped, Google Play absorbs the rest
app_store_total = len(APPS) * APP_STORE_CAP_PER_APP
play_total      = max(0, TARGET_TOTAL - app_store_total)
PLAY_TARGET_PER_APP = play_total // len(APPS)
print(f"App Store target: {APP_STORE_CAP_PER_APP}/app ({app_store_total} total)")
print(f"Google Play target: {PLAY_TARGET_PER_APP}/app ({PLAY_TARGET_PER_APP * len(APPS)} total)")
print(f"Combined target: {app_store_total + PLAY_TARGET_PER_APP * len(APPS)}")
for s in SOURCES: print("  ", s)

Reviews older than 2025-08-11 are excluded
App Store target: 500/app (1500 total)
Google Play target: 6166/app (18498 total)
Combined target: 19998
   ('google_play', 'Revolut')
   ('google_play', 'Klarna')
   ('google_play', 'Wise')
   ('app_store', 'Revolut')
   ('app_store', 'Klarna')
   ('app_store', 'Wise')


## 2. App Store adapter (RSS workaround)

`app-store-scraper` 0.3.5 breaks against Apple's authenticated `amp-api` endpoint. This subclass
reads from the public RSS feed instead. `build_url(page)` uses the `page=N/id=...` URL form, which is
the one that actually paginates (the `?page=N` form silently repeats the same 50 reviews).


In [4]:
from app_store_scraper import AppStore as _BaseAppStore

class RSSAppStore(_BaseAppStore):
    """AppStore subclass that reads reviews from the public RSS feed (amp-api now needs auth)."""
    _request_host = "itunes.apple.com"

    def build_url(self, page):
        return (f"https://itunes.apple.com/{self.country}/rss/customerreviews/"
                f"page={page}/id={self.app_id}/sortBy=mostRecent/json")

    def _parse_data(self, after):
        feed = self._response.json().get("feed", {})
        entries = feed.get("entry", [])
        if isinstance(entries, dict):          # a single entry comes back as a dict
            entries = [entries]
        for e in entries:
            review = {
                "id":        e.get("id", {}).get("label", ""),
                "userName":  e.get("author", {}).get("name", {}).get("label", ""),
                "rating":    int(e.get("im:rating", {}).get("label", 0)),
                "title":     e.get("title", {}).get("label", ""),
                "review":    e.get("content", {}).get("label", ""),
                "appVersion":e.get("im:version", {}).get("label", ""),
                "date":      datetime.strptime(e["updated"]["label"], "%Y-%m-%dT%H:%M:%S%z"),
            }
            if after and review["date"].replace(tzinfo=None) < after:
                continue
            self.reviews.append(review)
            self.reviews_count += 1
            self._fetched_count += 1

## 3. Normalisation helpers

Both scrapers return slightly different field names. These helpers map every review to one common
schema so the final dataset is a single tidy table.


In [5]:
COLS = ["platform", "app", "review_id", "user_name", "content", "score",
        "thumbs_up", "app_version", "reviewed_at", "reply", "replied_at"]

def norm_gplay(app, r):
    return {
        "platform": "google_play", "app": app,
        "review_id": r.get("reviewId"), "user_name": r.get("userName"),
        "content": r.get("content"), "score": r.get("score"),
        "thumbs_up": r.get("thumbsUpCount"),
        "app_version": r.get("reviewCreatedVersion") or r.get("appVersion"),
        "reviewed_at": pd.to_datetime(r.get("at"), utc=True, errors="coerce"),
        "reply": r.get("replyContent"), "replied_at": r.get("repliedAt"),
    }

def norm_ios(app, r):
    return {
        "platform": "app_store", "app": app,
        "review_id": r.get("id"), "user_name": r.get("userName"),
        "content": " | ".join(x for x in [r.get("title"), r.get("review")] if x),
        "score": r.get("rating"), "thumbs_up": None,
        "app_version": r.get("appVersion"),
        "reviewed_at": pd.to_datetime(r.get("date"), utc=True, errors="coerce"),
        "reply": None, "replied_at": None,
    }

def load_existing(path):
    """Return (rows, seen_ids) loaded from a previously saved checkpoint CSV."""
    if os.path.exists(path) and os.path.getsize(path) > 0:
        df = pd.read_csv(path)
        if len(df):
            return df.to_dict("records"), set(df["review_id"].dropna().astype(str))
    return [], set()

def save_progress(path, rows):
    pd.DataFrame(rows, columns=COLS).to_csv(path, index=False)

## 4. Scrapers

Each scraper pages through its source, keeps only reviews on/after `CUTOFF`, and **saves progress to
CSV after every page** so a long run can be resumed (`seen` de-duplicates on restart).

### Google Play
`google_play_scraper.reviews()` returns `(reviews, continuation_token)`; we pass the token back to walk
deeper pages (199 reviews/request). Stops when the target is met or a page goes older than the cutoff.


In [6]:
def scrape_google_play(app, target, cutoff):
    package = APPS[app]["gplay"]
    path = os.path.join(RAW_DIR, f"google_play_{app}.csv")
    rows, seen = load_existing(path)
    print(f"[google_play/{app}] starting (have {len(rows)} rows)")

    token = None
    while len(rows) < target:
        try:
            page, token = gp_reviews(package, lang=LANG, country=COUNTRY,
                                     sort=GPSort.NEWEST, count=199,
                                     continuation_token=token)
        except Exception as e:
            print("  ! page error, retrying in 5s:", e)
            time.sleep(5)
            continue
        if not page:
            break

        new = []
        for r in page:
            rid = r.get("reviewId")
            if rid in seen:
                continue
            seen.add(rid)
            date = pd.to_datetime(r.get("at"), utc=True, errors="coerce")
            if date is not None and date.tz_localize(None) < cutoff:
                continue
            new.append(norm_gplay(app, r))

        rows.extend(new)
        save_progress(path, rows)
        print(f"  page: {len(page)} fetched, +{len(new)} kept, total {len(rows)}/{target}")

        if token is None:
            print("  no more pages available")
            break
        if page[-1]["at"].date() < cutoff.date():
            print("  reached reviews older than cutoff")
            break
        time.sleep(SLEEP_GPLAY)
    return rows

### Apple App Store (RSS)
The RSS feed exposes 10 pages of 50 reviews per app. Each page is fetched, filtered and checkpointed
independently, so the loop stops at the RSS cap or when the feed runs out.


In [7]:
def scrape_app_store(app, target, cutoff):
    cfg = APPS[app]
    path = os.path.join(RAW_DIR, f"app_store_{app}.csv")
    rows, seen = load_existing(path)
    print(f"[app_store/{app}] starting (have {len(rows)} rows)")

    store = RSSAppStore(country=COUNTRY, app_name=cfg["ios_name"], app_id=cfg["ios_id"])
    headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)", "Accept": "application/json"}

    page = 1
    while len(rows) < target and page <= 10:
        before = len(rows)
        try:
            store.reviews = []
            store._get(store.build_url(page), headers=headers)
            store._parse_data(cutoff)
        except Exception as e:
            print(f"  ! page {page} error, retrying in 5s:", e)
            time.sleep(5)
            continue

        new = []
        for r in store.reviews:
            rid = r.get("id")
            if rid in seen:
                continue
            seen.add(rid)
            new.append(norm_ios(app, r))

        rows.extend(new)
        save_progress(path, rows)
        print(f"  page {page}: {len(store.reviews)} fetched, +{len(new)} kept, total {len(rows)}/{target}")

        if not store.reviews:
            print("  no more reviews available (RSS feed exhausted)")
            break
        page += 1
        time.sleep(SLEEP_RSS)
    return rows

## 5. Run the scraper

> **`SMOKE_TEST = True` (default):** grabs ~10 reviews per source to verify the whole pipeline.
>
> **`SMOKE_TEST = False`:** collects ~20,000 reviews over the past 12 months. This makes thousands of
> requests and takes **30–90+ minutes**. Progress is checkpointed to `data/raw/*.csv` after every page,
> so you can stop and re-run the cell to continue where it left off.


In [8]:
all_rows = []
for platform, app in SOURCES:
    if SMOKE_TEST:
        target = 10
    elif platform == "google_play":
        target = PLAY_TARGET_PER_APP
    else:
        target = APP_STORE_CAP_PER_APP

    if platform == "google_play":
        rows = scrape_google_play(app, target, CUTOFF)
    else:
        rows = scrape_app_store(app, target, CUTOFF)
    all_rows.extend(rows)
    print(f"== {platform}/{app} done: {len(rows)} rows\n")

print(f"\nCollected {len(all_rows)} rows total")

[google_play/Revolut] starting (have 6169 rows)
== google_play/Revolut done: 6169 rows

[google_play/Klarna] starting (have 6169 rows)
== google_play/Klarna done: 6169 rows

[google_play/Wise] starting (have 6169 rows)
== google_play/Wise done: 6169 rows

[app_store/Revolut] starting (have 500 rows)


2026-08-11 10:16:32,245 [INFO] Base - Initialised: RSSAppStore('gb', 'revolut', 932493382)
2026-08-11 10:16:32,246 [INFO] Base - Ready to fetch reviews from: https://apps.apple.com/gb/app/revolut/id932493382


== app_store/Revolut done: 500 rows

[app_store/Klarna] starting (have 500 rows)


2026-08-11 10:16:32,922 [INFO] Base - Initialised: RSSAppStore('gb', 'klarna', 1115120118)
2026-08-11 10:16:32,922 [INFO] Base - Ready to fetch reviews from: https://apps.apple.com/gb/app/klarna/id1115120118


== app_store/Klarna done: 500 rows

[app_store/Wise] starting (have 500 rows)


2026-08-11 10:16:33,792 [INFO] Base - Initialised: RSSAppStore('gb', 'wise', 612261027)
2026-08-11 10:16:33,793 [INFO] Base - Ready to fetch reviews from: https://apps.apple.com/gb/app/wise/id612261027


== app_store/Wise done: 500 rows


Collected 20007 rows total


## 6. Combine, clean and save

Merges every checkpoint file into a single tidy dataset, drops duplicate review IDs and any rows whose
date fell outside the window, then saves the final CSV.


In [9]:
frames = []
for platform, app in SOURCES:
    path = os.path.join(RAW_DIR, f"{platform}_{app}.csv")
    if os.path.exists(path) and os.path.getsize(path) > 0:
        frames.append(pd.read_csv(path))

if not frames:
    print("No data yet - run the scraping cell above first.")
else:
    df = pd.concat(frames, ignore_index=True)

    df = df.dropna(subset=["reviewed_at"])
    df["reviewed_at"] = pd.to_datetime(df["reviewed_at"], utc=True, errors="coerce")
    df = df[df["reviewed_at"].dt.tz_localize(None) >= CUTOFF]
    df = df.drop_duplicates(subset=["platform", "review_id"]).reset_index(drop=True)

    out = os.path.join(DATA_DIR, "reviews_all.csv")
    df.to_csv(out, index=False)
    print("Saved", out)
    print("Total rows:", len(df))

Saved data/reviews_all.csv
Total rows: 20007


## 7. Quick overview

Summary statistics: reviews per app and per platform, rating distribution, and the actual date span.


In [10]:
if 'df' in dir():
    print("=== Reviews per app ===")
    print(df.groupby("app", observed=True).size().to_string())
    print("\n=== Reviews per platform ===")
    print(df.groupby("platform", observed=True).size().to_string())
    print("\n=== Rating distribution (score) ===")
    print(df.groupby("score", observed=True).size().sort_index().to_string())
    print("\n=== Date range ===")
    print("Earliest:", df["reviewed_at"].min().date(), "| Latest:", df["reviewed_at"].max().date())
    print("\n=== Preview ===")
    display(df.head(10))

=== Reviews per app ===
app
Klarna     6669
Revolut    6669
Wise       6669

=== Reviews per platform ===
platform
app_store       1500
google_play    18507

=== Rating distribution (score) ===
score
1     3651
2      554
3      570
4     1001
5    14231

=== Date range ===
Earliest: 2026-03-25 | Latest: 2026-08-09

=== Preview ===


,platform,app,review_id,user_name,content,score,thumbs_up,app_version,reviewed_at,reply,replied_at
0,google_play,Revolut,287aabe8-adb2-4315-8a1c-6a9d1c88a14d,chris mabey,great app,5,0.0,10.142,2026-08-09 21:07:12+00:00,NaN,NaN
1,google_play,Revolut,ad5e36d9-290c-4a75-b71e-82231c54f3f1,PAULO AGNEL VAS,excellent,4,0.0,10.141,2026-08-09 20:29:10+00:00,NaN,NaN
2,google_play,Revolut,4adfec84-a5f7-4c16-8758-2c83ae54675b,班KAMI,"Been using for many years, great experience ti...",5,0.0,10.141,2026-08-09 19:28:00+00:00,NaN,NaN
3,google_play,Revolut,e3e94c14-3b4b-410a-875c-549e0a21dd15,Ilia K,blocked account without any explanation after ...,1,0.0,10.141,2026-08-09 18:55:22+00:00,Hi. Sometimes we need to take extra steps to e...,2026-08-09 19:16:03
4,google_play,Revolut,65f46eb8-e613-4261-9af4-51ec35a30cdc,Sam Osagiede,great app.,5,0.0,10.141,2026-08-09 18:44:05+00:00,NaN,NaN
5,google_play,Revolut,d8acfc6c-3b3e-442a-8284-d8894b664ee4,Josianne Pace,good,5,0.0,10.142,2026-08-09 18:26:47+00:00,NaN,NaN
6,google_play,Revolut,1ef49d2f-28ac-4132-ad4a-02536b61739f,Massimo Casoni,complete and easy to use,5,0.0,10.141,2026-08-09 17:53:03+00:00,NaN,NaN
7,google_play,Revolut,8d6ae521-7cb7-44ec-b0a8-6371380ae0e7,Peter Lucz,all good except one but very important securit...,4,0.0,10.142,2026-08-09 17:29:07+00:00,NaN,NaN
8,google_play,Revolut,80043c10-4aa1-4664-901c-05abf044368f,sorna kavoosi,Fastest banking app I've worked with sofar,5,0.0,10.139,2026-08-09 17:08:56+00:00,NaN,NaN
9,google_play,Revolut,c695263f-972c-43de-ae55-f2a4a0730847,Ebins Sunny,excellent,5,0.0,10.141,2026-08-09 16:46:49+00:00,NaN,NaN
